# **Pose Detection Testing NB**

In [1]:
from transformers import pipeline

/Users/charlesberman/Desktop/Plongeon-Project/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1120 16:40:22.666000 36700 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Foot keypoint comparison: Sapiens2 vs RTMW vs BlazePose
Runs every model in `models` on every PNG in `pictures/` and writes a side-by-side comparison to `pose_comparison.pdf`.

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import torch
from huggingface_hub import hf_hub_download
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
from PIL import Image, ImageOps
from rtmlib import RTMPose
from transformers import AutoImageProcessor, Sapiens2ForPoseEstimation

# List of all images in pictures/ (skipping visualizations written by earlier versions of this notebook)
folder = Path("pictures/")
images = sorted(p for p in folder.glob("*.png") if not p.stem.endswith("_heels"))
# Extra clockwise rotation (degrees) after EXIF orientation is applied, if the person still is not upright.
rotate_cw = 0

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

In [ ]:
# TODO: add all model ids that afsaneh sent
# Each entry: kind (selects the inference code in pose_detect), processor, model,
# keypoints (name -> index in that model's skeleton) and score_threshold (below it a point counts as low confidence;
# scores are on different scales per model, so thresholds are per model).
SAPIENS_ID = "facebook/sapiens2-pose-0.4b"
RTMW_ID = "DavidPagnon/rtmlib_models"
BLAZE_ID = "unity/sentis-blaze-pose"

models = {
    SAPIENS_ID: {
        "kind": "sapiens2",
        "processor": AutoImageProcessor.from_pretrained(SAPIENS_ID),
        "model": Sapiens2ForPoseEstimation.from_pretrained(SAPIENS_ID).to(device).eval(),
        # Goliath 308 ordering ("goliath" keypoint_info table in sapiens/pose/configs/_base_/keypoints308.py)
        "keypoints": {
            "left_ankle": 13, "right_ankle": 14,
            "left_big_toe": 15, "left_small_toe": 16, "left_heel": 17,
            "right_big_toe": 18, "right_small_toe": 19, "right_heel": 20,
        },
        "score_threshold": 0.3,  # heatmap peak, roughly 0..1
    },
    RTMW_ID: {
        "kind": "rtmw",
        "processor": None,  # rtmlib does its own affine crop + normalization and SimCC decoding
        "model": RTMPose(
            onnx_model=hf_hub_download(
                RTMW_ID,
                "mmpose/rtmw/onnx_sdk/rtmw-x_simcc-cocktail14_pt-ucoco_270e-384x288-f840f204_20231122.onnx",
                repo_type="dataset",
            ),
            model_input_size=(288, 384),  # (w, h)
            backend="onnxruntime",
            device="cpu",
        ),
        # COCO-WholeBody 133 ordering
        "keypoints": {
            "left_ankle": 15, "right_ankle": 16,
            "left_big_toe": 17, "left_small_toe": 18, "left_heel": 19,
            "right_big_toe": 20, "right_small_toe": 21, "right_heel": 22,
        },
        "score_threshold": 3.0,  # SimCC confidence, unnormalized (good points are ~7-9)
    },
    BLAZE_ID: {
        "kind": "blazepose",
        "processor": None,  # preprocessing is in the BlazePose helper cell below
        "model": {
            "detector": ort.InferenceSession(hf_hub_download(BLAZE_ID, "models/pose_detection.onnx")),
            "landmarker": ort.InferenceSession(hf_hub_download(BLAZE_ID, "models/pose_landmarks_detector_heavy.onnx")),
        },
        # BlazePose 33 ordering; it has one toe point per foot ("foot_index") and no small toe
        "keypoints": {
            "left_ankle": 27, "right_ankle": 28,
            "left_heel": 29, "right_heel": 30,
            "left_foot_index": 31, "right_foot_index": 32,
        },
        "score_threshold": 0.5,  # sigmoid(visibility)
    },
}

In [ ]:
# BlazePose (MediaPipe pose landmarker, heavy) from ONNX, following Unity's BlazeDetectionSample/Pose:
# 224x224 detector finds the person and two alignment points (mid-hip, body extent),
# then a rotated 256x256 crop around them goes through the landmarker (33 landmarks x [x, y, z, visibility, presence]).

def _blaze_anchors(size: int = 224) -> np.ndarray:
    """SSD anchor centers of the pose detector (strides 8/16/32, fixed size) -> (2254, 2) in 0..1."""
    centers = []
    for stride, per_loc in [(8, 2), (16, 2), (32, 6)]:
        n = size // stride
        for y in range(n):
            for x in range(n):
                centers += [((x + 0.5) / n, (y + 0.5) / n)] * per_loc
    return np.array(centers, np.float32)


BLAZE_ANCHORS = _blaze_anchors()


def _sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -100, 100)))


def blazepose_infer(sessions: dict, rgb: np.ndarray, work_size: int = 1024) -> tuple[np.ndarray, np.ndarray]:
    """Run detector + landmarker on an RGB uint8 image. Returns (33, 2) pixel coords and (33,) visibility scores."""
    h, w = rgb.shape[:2]
    # Work on a downscaled copy so the crops below don't alias
    s_work = min(1.0, work_size / max(h, w))
    small = cv2.resize(rgb, None, fx=s_work, fy=s_work, interpolation=cv2.INTER_AREA)
    sh, sw = small.shape[:2]

    # Detector: letterbox to a centered square, resize to 224, values in [-1, 1]
    size = max(sh, sw)
    ox, oy = (size - sw) // 2, (size - sh) // 2
    square = np.zeros((size, size, 3), np.uint8)
    square[oy:oy + sh, ox:ox + sw] = small
    det_in = cv2.resize(square, (224, 224), interpolation=cv2.INTER_AREA).astype(np.float32)[None] / 127.5 - 1
    boxes, scores = sessions["detector"].run(None, {"input_1": det_in})
    best = int(np.argmax(scores[0, :, 0]))
    # Alignment keypoints (offsets in 224-px tensor space from the anchor), mapped to `small` pixels
    kp = (BLAZE_ANCHORS[best] * 224 + boxes[0, best, 4:8].reshape(2, 2)) * (size / 224) - [ox, oy]

    # Landmarker crop: centered on mid-hip, radius 1.25 * |hip -> extent|, rotated so the body points up
    d = kp[1] - kp[0]
    radius = 1.25 * np.linalg.norm(d)
    ux, uy = d / np.linalg.norm(d)
    lin = radius / 128 * np.array([[-uy, -ux], [ux, -uy]], np.float32)  # crop -> image; crop "up" maps to d
    t = kp[0] - lin @ np.array([128, 128], np.float32)
    M = np.hstack([lin, t[:, None]]).astype(np.float32)
    crop = cv2.warpAffine(small, M, (256, 256), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP)

    out = sessions["landmarker"].run(None, {"input_1": crop.astype(np.float32)[None] / 255})
    lm = out[0].reshape(-1, 5)[:33]
    xy = (lm[:, :2] @ lin.T + t) / s_work
    return xy, _sigmoid(lm[:, 3])

In [ ]:
def to_original(x: float, y: float, W: int, H: int) -> tuple[float, float]:
    """Map a point in the upright image back to the original (unrotated, W x H) image."""
    k = rotate_cw % 360
    if k == 0:
        return x, y
    if k == 90:   # upright = orig rotated CW 90: (x, y) -> orig (y, H-1-x)
        return y, H - 1 - x
    if k == 180:
        return W - 1 - x, H - 1 - y
    if k == 270:
        return W - 1 - y, x
    raise ValueError("rotation must be a multiple of 90")


def load_image(img_path: Path) -> tuple[Image.Image, Image.Image]:
    """Return (original with EXIF orientation applied, upright copy rotated by rotate_cw)."""
    orig = ImageOps.exif_transpose(Image.open(img_path)).convert("RGB")
    return orig, orig.rotate(-rotate_cw, expand=True)


def pose_detect(model_id: str, img_path: Path, verbose: bool = True) -> dict[str, tuple[float, float, float]]:
    """
    Run model `model_id` (a key of `models`) on one image.

    Returns {keypoint_name: (x, y, score)} for that model's `keypoints`, in pixel coordinates of the
    EXIF-oriented original image. Top-down models (Sapiens2, RTMW) use the whole image as the person box;
    BlazePose runs its own person detector.
    """
    entry = models[model_id]
    orig, upright = load_image(img_path)
    W, H = orig.size
    uw, uh = upright.size

    if entry["kind"] == "sapiens2":
        boxes = [[[0.0, 0.0, float(uw), float(uh)]]]  # COCO xywh
        processor = entry["processor"]
        inputs = processor(images=upright, boxes=boxes, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = entry["model"](**inputs)
        person = processor.post_process_pose_estimation(outputs, boxes=boxes)[0][0]
        kps, scores = person["keypoints"].cpu().numpy(), person["scores"].cpu().numpy()
    elif entry["kind"] == "rtmw":
        bgr = cv2.cvtColor(np.asarray(upright), cv2.COLOR_RGB2BGR)  # rtmlib convention (cv2.imread order)
        kps, scores = entry["model"](bgr, bboxes=[[0, 0, uw, uh]])  # xyxy
        kps, scores = kps[0], scores[0]
    elif entry["kind"] == "blazepose":
        kps, scores = blazepose_infer(entry["model"], np.asarray(upright))
    else:
        raise ValueError(f"unknown model kind {entry['kind']!r}")

    results = {}
    for name, idx in entry["keypoints"].items():
        x, y = to_original(float(kps[idx][0]), float(kps[idx][1]), W, H)
        results[name] = (x, y, float(scores[idx]))

    if verbose:
        print(f"Image {img_path} ({W}x{H}), model {model_id}")
        for name, (x, y, s) in results.items():
            print(f"  {name:16s} x={x:8.1f}  y={y:8.1f}  score={s:.3f}")
    return results

In [ ]:
# Run every model on every picture, then write one PDF page per picture with one column per model:
# full image on top, zoom on the feet below (same crop for all models so they can be compared directly).
colors = {
    "left_heel": "#00c000", "right_heel": "#ff8000",
    "left_ankle": "#0078ff", "right_ankle": "#ff00c8",
    "left_big_toe": "#00e0e0", "left_small_toe": "#e0d000",
    "right_big_toe": "#ff0000", "right_small_toe": "#a020f0",
    "left_foot_index": "#00e0e0", "right_foot_index": "#ff0000",  # BlazePose toe tip, same colors as big toes
}

all_results = {img: {mid: pose_detect(mid, img, verbose=False) for mid in models} for img in images}


def is_confident(mid, x, y, s, W, H):
    return s >= models[mid]["score_threshold"] and 0 <= x < W and 0 <= y < H


def feet_box(results_by_model, W, H, margin=0.35):
    """Bounding box (x0, y0, x1, y1) of all models' confident foot points, padded; bottom half if none."""
    pts = [(x, y) for mid, res in results_by_model.items() for x, y, s in res.values() if is_confident(mid, x, y, s, W, H)]
    if not pts:
        return 0, H // 2, W, H
    xs, ys = np.array(pts).T
    pad = margin * max(xs.max() - xs.min(), ys.max() - ys.min(), 0.05 * max(W, H))
    return (int(max(0, xs.min() - pad)), int(max(0, ys.min() - pad)),
            int(min(W, xs.max() + pad)), int(min(H, ys.max() + pad)))


def draw_points(ax, mid, res, W, H, zoom):
    for name, (x, y, s) in res.items():
        ok = is_confident(mid, x, y, s, W, H)
        ax.plot(x, y, marker="o" if ok else "X", ms=9 if zoom else 5, mfc=colors[name] if ok else "none",
                mec="white" if ok else colors[name], mew=1.0 if ok else 1.5, ls="none")
        if zoom and 0 <= x < W and 0 <= y < H:
            ax.annotate(f"{s:.2f}", (x, y), xytext=(6, 6), textcoords="offset points", fontsize=7,
                        color="white", bbox=dict(boxstyle="round,pad=0.15", fc=colors[name], ec="none", alpha=0.8))


out_pdf = Path("pose_comparison.pdf")
with PdfPages(out_pdf) as pdf:
    for img_path, results_by_model in all_results.items():
        orig, _ = load_image(img_path)
        W, H = orig.size
        preview = orig.copy()
        preview.thumbnail((700, 700))
        x0, y0, x1, y1 = feet_box(results_by_model, W, H)
        zoom_img = orig.crop((x0, y0, x1, y1))
        zoom_img.thumbnail((800, 800))

        fig, axes = plt.subplots(2, len(models), figsize=(5.5 * len(models), 11), squeeze=False,
                                 gridspec_kw={"height_ratios": [1, 1]})
        for col, (mid, res) in enumerate(results_by_model.items()):
            top, bottom = axes[0, col], axes[1, col]
            top.imshow(preview, extent=(0, W, H, 0))
            top.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="yellow", lw=1, ls="--"))
            draw_points(top, mid, res, W, H, zoom=False)
            top.set_title(mid, fontsize=10)
            bottom.imshow(zoom_img, extent=(x0, x1, y1, y0))
            draw_points(bottom, mid, res, W, H, zoom=True)
            bottom.set_xlim(x0, x1); bottom.set_ylim(y1, y0)
            for ax in (top, bottom):
                ax.set_xticks([]); ax.set_yticks([])

        names = list(dict.fromkeys(n for res in results_by_model.values() for n in res))
        handles = [Line2D([], [], marker="o", ls="none", mfc=colors[n], mec="white", ms=8, label=n) for n in names]
        handles.append(Line2D([], [], marker="X", ls="none", mfc="none", mec="gray", ms=8,
                              label="below model's score_threshold"))
        fig.legend(handles=handles, loc="lower center", ncol=6, fontsize=8, frameon=False)
        fig.suptitle(f"{img_path.name}  ({W}x{H})  —  numbers are each model's own score "
                     "(Sapiens2 ~0-1, RTMW unnormalized SimCC, BlazePose visibility 0-1)", fontsize=11)
        fig.tight_layout(rect=(0, 0.04, 1, 0.97))
        pdf.savefig(fig, dpi=100)  # embedded images are resampled to this dpi; raise for sharper zooms
        plt.close(fig)

print(f"Saved {len(all_results)} pages to {out_pdf}")